# LASSER, option 1: learned co-expression graph with the GEARS batching fix

**Why this study.** In upstream GEARS the co-expression GNN only reaches the first cell of each batch: it stacks
32 cells' gene embeddings but uses a single-cell `edge_index`. So the co-expression graph barely affects
predictions, and in study session 1 GEARS and LASSER came out the same. `fix_coexpress_batching=True` (opt-in;
off = upstream GEARS, bit-for-bit) gives every cell its graph messages. This study asks whether a **learned**
graph beats GEARS's Pearson graph **once the graph can act**, and checks that against two controls.

| Tier | Jobs (10 epochs each) | Question |
|---|---|---|
| 1 | `GEARS-fix` vs `LASSER-fix`, splits 1–3 | Does the learned graph help? |
| 2 | `LASSER-random-fix`, splits 1–3 | Is it the learned *partners*, or just extra edges? (same in-degrees and weights) |
| 3 | `GEARS-kNN8-fix`, splits 1–3 | Is it more than a denser Pearson graph? |
| 4 | `GEARS` (no fix), splits 1–3 | What does the fix itself change? |
| 5 | training-seed replicates (`_ts101`) of tier 1 | How large is run-to-run noise? |
| 6 | splits 4–5 for tiers 1–3 | More splits |

**Budget.** About 50 min per run on one GPU (10 epochs), so one **10-hour** session finishes tiers 1–2 and
part of tier 3 (about 10–11 runs). Jobs run in this priority order, interleaved by split, so whatever finishes
is a complete paired set. Later sessions resume where this one stopped.

**Kaggle settings:** GPU (one T4 or P100 is enough; RAM allows only one worker), **Internet on**. Run with
*Save Version → Save & Run All (Commit)*. To resume: add this notebook's previous output as input and set
`RESUME_FROM`.

Runs from option 1 use 10 epochs everywhere, so they are not compared with the 20-epoch runs of study session 1.

## A. Setup and experiments

In [ ]:
# 1. Dependencies (GEARS is vendored inside lasser/).
!pip install -q torch_geometric scanpy pytest

In [ ]:
# 2. Code.
REPO_URL = "https://github.com/LegendaryDarkKnight/LASSER"
REPO_DIR = "/kaggle/working/lasser-repo"
!rm -rf {REPO_DIR} /kaggle/working/lasser
!git clone -q {REPO_URL} {REPO_DIR}
!mv {REPO_DIR}/lasser /kaggle/working/lasser
!git -C {REPO_DIR} log -1 --oneline

In [ ]:
# 3. Settings.
import os, shutil, time
T0 = time.time()
SESSION_HOURS = 10.0                # your limit for this session
ANALYSIS_RESERVE_HOURS = 0.6        # kept free at the end for analysis + saving outputs
EPOCHS = 10
STUDY_DIR = "/kaggle/working/fix_study"
DATA_DIR = "/tmp/lasser_data"       # not saved as output (several GB); re-downloaded each session
RESUME_FROM = None                  # e.g. "/kaggle/input/<previous-version>/fix_study"

os.makedirs(STUDY_DIR, exist_ok=True)
if RESUME_FROM:
    shutil.copytree(RESUME_FROM, STUDY_DIR, dirs_exist_ok=True)
    shutil.rmtree(f"{STUDY_DIR}/runs/claims", ignore_errors=True)
    print("resumed from", RESUME_FROM)
!nvidia-smi --query-gpu=index,name,memory.total --format=csv
!free -g | head -2

In [ ]:
# 4. Fast tests (seconds, no data): parser, random-edge control, study tooling, batching fix, replicates.
!cd /kaggle/working && python -m pytest {REPO_DIR}/tests/test_05_parsing.py {REPO_DIR}/tests/test_06_study.py {REPO_DIR}/tests/test_07_fix.py -q -p no:cacheprovider --tb=short 2>&1 | grep -vE "Warning|warn|^  |^$"

In [ ]:
# 5. Experiment table (priority order) and what is already done.
import pandas as pd
from lasser.study import ARM_DESCRIPTIONS, FIX_PLAN, make_jobs, study_status, expected_minutes, default_minutes

BASE = dict(dataset="norman", split="simulation", epochs=EPOCHS, device="cuda",
            data_dir=DATA_DIR, out_dir=f"{STUDY_DIR}/runs", cache_dir=f"{STUDY_DIR}/cache", repo_dir=REPO_DIR)
jobs = make_jobs(FIX_PLAN, BASE)
status = study_status(jobs)
est = expected_minutes(BASE["out_dir"], default_minutes(EPOCHS))
left = SESSION_HOURS - ANALYSIS_RESERVE_HOURS - (time.time() - T0) / 3600
print(f"{len(jobs)} jobs, {int(status.done.sum())} done; ~{est:.0f} min each -> ~{int(left * 60 // est)} fit in this session")
display(status[["job_id", "arm", "seed", "train_seed", "done"]].assign(
    what=lambda d: d.arm.map(ARM_DESCRIPTIONS)).head(40))

In [ ]:
# 6. Data, splits and GEARS graphs, built once (~15 min the first time).
from lasser.study import prepare_data
if not study_status(jobs).done.all():
    prepare_data(jobs)

In [ ]:
# 7. Run jobs in priority order until the budget is used (one worker; progress every 10 min).
from lasser.study import launch
budget = SESSION_HOURS - ANALYSIS_RESERVE_HOURS - (time.time() - T0) / 3600
if not study_status(jobs).done.all():
    launch(jobs, budget_hours=budget, poll_minutes=10)
status = study_status(jobs)
print(f"{int(status.done.sum())}/{len(status)} jobs done")
if os.path.exists(f"{BASE['out_dir']}/failures.csv"):
    display(pd.read_csv(f"{BASE['out_dir']}/failures.csv").tail(5))
status[status.done][["job_id", "test_pearson_delta", "train_minutes"]]

## B. Analysis

Works on whatever has finished, and can be rerun alone (CPU) on a previous output by pointing `STUDY_DIR` at it.

- The **unit of replication is the split**: training-seed replicates are averaged within a split first
  (`unit="split"`). Tables with `unit="run"` treat each (split, training seed) pair as a unit, with more pairs but
  correlated within a split.
- Arms with the same split and training seed share initialisation and batch order, so their difference isolates
  the graph.
- Improvement > 0 means better, for every metric (the sign is handled per metric).

In [ ]:
# 8. Load runs and the reference baselines (no perturbation, additive).
import numpy as np
from lasser import analysis as A
from lasser import figures as F
F.use_fix_colors()
pd.set_option("display.max_columns", 60); pd.set_option("display.width", 250)
RESULTS_DIR, FIG_DIR = f"{STUDY_DIR}/results", f"{STUDY_DIR}/figures"
runs = A.load_runs(f"{STUDY_DIR}/runs")
long = A.metrics_long(runs)
H5AD = f"{DATA_DIR}/norman/perturb_processed.h5ad"
if os.path.exists(H5AD):
    long = pd.concat([long, A.paper_reference_baselines(runs, H5AD)], ignore_index=True)
ARMS = [a for a in ["GEARS", "GEARS-fix", "LASSER-fix", "GEARS-kNN8-fix", "LASSER-random-fix"] if a in set(runs.arm)]
REFS = [a for a in ["No-perturb", "Additive"] if a in set(long[long.family == "paper"].arm)]
SCOPES = ["overall"] + A.SUBGROUPS
display(runs.pivot_table(index="arm", columns=["seed", "rep"], values="test_pearson_delta").reindex(ARMS).round(3))

In [ ]:
# 9. Table 1: headline metrics (mean ± sd over splits; replicates averaged within split).
t1 = A.summary_table(long, A.HEADLINE, scopes=["overall"], arms=REFS + ARMS)
t1_sub = A.summary_table(long, ["pearson_delta", "mse_top20_de", "frac_opposite_top20_de"], scopes=A.SUBGROUPS,
                         arms=REFS + ARMS)
display(t1); display(t1_sub)

In [ ]:
# 10. Table 2: the paired comparisons that carry the claim.
COMPARISONS = [("GEARS-fix", "LASSER-fix"),          # learned vs Pearson graph (both fixed)
               ("LASSER-random-fix", "LASSER-fix"),  # learned vs random partners, same degrees/weights
               ("GEARS-kNN8-fix", "LASSER-fix"),     # learned vs density-matched Pearson
               ("GEARS", "GEARS-fix")]               # what the batching fix itself does
show = ["arm", "ref", "unit", "scope", "metric", "n_splits", "ref_mean", "arm_mean", "improvement", "ci95_low",
        "ci95_high", "splits_better", "p_t", "p_wilcoxon"]
t2 = pd.concat([A.paired_tests(long, ref, [arm], A.HEADLINE, scopes=SCOPES, unit=u)
                for ref, arm in COMPARISONS if {ref, arm} <= set(ARMS) for u in ("split", "run")], ignore_index=True)
if len(t2):
    t2["p_t_holm"] = np.nan
    for u in ("split", "run"):
        m = t2.unit == u
        t2.loc[m, "p_t_holm"] = A._holm(t2.loc[m, "p_t"].to_numpy())
    display(t2[(t2.scope == "overall") & (t2.unit == "split")][show + ["p_t_holm"]].round(4))
    display(t2[(t2.scope == "overall") & (t2.unit == "run")][show + ["p_t_holm"]].round(4))
    display(t2[(t2.scope != "overall") & (t2.metric == "pearson_delta") & (t2.unit == "split")][show].round(4))

In [ ]:
# 11. Table 3: per-perturbation paired analysis (pairs = split x training seed x perturbation;
#     bootstrap resamples splits, then pairs within them).
pp = A.per_perturbation(runs)
t3 = pd.concat([A.paired_perturbation(pp, ref, arm, ["pearson_delta", "mse_top20_de", "frac_opposite_top20_de"])
                for ref, arm in COMPARISONS if {ref, arm} <= set(ARMS)], ignore_index=True)
display(t3.round(4))

In [ ]:
# 12. Run-to-run noise: same arm, same split, two training seeds. An effect smaller than this can't be claimed.
rep_runs = runs[runs.arm.isin(["GEARS-fix", "LASSER-fix"])]
w = rep_runs.pivot_table(index=["arm", "seed"], columns="rep", values="test_pearson_delta")
if w.shape[1] >= 2:
    w["abs_diff"] = (w.iloc[:, 0] - w.iloc[:, 1]).abs()
    display(w.round(4))
    print(f"mean |difference| between training seeds: {w['abs_diff'].mean():.4f} Pearson Δ")
else:
    print("no training-seed replicates yet (tier 5)")

In [ ]:
# 13. Where does it help? Output genes isolated vs connected in GEARS's A0 graph.
gg = A.gene_group_metrics(runs)
t5 = A.gene_group_paired(gg, "GEARS-fix", [a for a in ["LASSER-fix", "LASSER-random-fix", "GEARS-kNN8-fix"]
                                          if a in ARMS]) if "GEARS-fix" in ARMS else pd.DataFrame()
display(t5.round(4))

In [ ]:
# 14. The learned graph: checks, stability across splits, overlap with the Pearson kNN graph, consensus edges.
gt = A.graph_table(runs)
display(gt.drop_duplicates(["arm", "seed"]).groupby("arm")[[c for c in ["num_edges", "isolated_genes",
        "degree_median", "h2_hit_added", "h2_hit_random", "h2_hit_a0", "link_auc", "link_auc_raw", "jaccard_to_a0"]
        if c in gt]].agg(["mean", "std"]).round(3))
consensus = pd.DataFrame()
if "LASSER-fix" in ARMS:
    jac, counts = A.edge_stability(runs, "LASSER-fix")
    off = jac.to_numpy(dtype=float)[~np.eye(len(jac), dtype=bool)]
    print(f"mean Jaccard of new-edge sets between splits: {np.nanmean(off):.3f}")
    display(A.overlap_with_knn8(runs, "LASSER-fix"))
    consensus = A.consensus_edges(runs, "LASSER-fix", min_splits=min(3, runs[runs.arm == "LASSER-fix"].seed.nunique()))
    display(consensus.head(30))

In [ ]:
# 15. Figures (PDF + PNG).
paths = []
if ARMS:
    paths += F.fig_headline(long, REFS + ARMS, FIG_DIR)
    paths += F.fig_subgroups(long, ARMS, FIG_DIR)
if {"GEARS-fix", "LASSER-fix"} <= set(ARMS):
    paths += F.fig_per_perturbation(pp, FIG_DIR, ref="GEARS-fix", arm="LASSER-fix")
    paths += F.fig_gene_groups(t5, FIG_DIR, arm="LASSER-fix")
    paths += F.fig_graphs(gt, runs, FIG_DIR, a0_arm="GEARS-fix", knn_arm="GEARS-kNN8-fix",
                          lasser_arm="LASSER-fix", random_arm="LASSER-random-fix")
if len(t2):
    paths += F.fig_forest(t2[t2.unit == "split"], FIG_DIR, "fig_forest_fix", "Paired improvement (splits)")
paths += F.fig_training(A.training_curves(runs), ARMS, FIG_DIR)
from IPython.display import Image, display as _d
for p in paths:
    if p.endswith(".png"):
        _d(Image(p, width=900))

In [ ]:
# 16. Export and zip (download /kaggle/working/lasser_fix_study_results.zip).
A.export_tables({"table1_headline": t1, "table1_subgroups": t1_sub, "table2_paired": t2,
                 "table3_paired_perturbations": t3, "table5_gene_groups": t5, "graph_stats": gt,
                 "consensus_edges": consensus}, RESULTS_DIR)
!cd {STUDY_DIR} && zip -qr /kaggle/working/lasser_fix_study_results.zip results figures runs/runs.csv runs/jobs.json runs/logs
print(sorted(os.listdir(RESULTS_DIR)))

## C. Reading the results

- **The claim** is `LASSER-fix > GEARS-fix` **and** `LASSER-fix > LASSER-random-fix` (and ideally
  `> GEARS-kNN8-fix`) on the paired tests, with an improvement larger than the run-to-run noise in cell 12.
- If `GEARS-fix` vs `GEARS` is large, the fix itself is a finding: published GEARS barely uses its co-expression
  graph.
- With 3 splits, p-values are weak by construction (paired *t*-test, n = 3). Report effect sizes with CIs,
  splits-better counts and the per-perturbation analysis, and add splits 4–5 (tier 6) in a later session before
  submission.